# SLD pipeline v3

**Part A** component classifier: data, augmentation, live training, curves, ROC / residuals, test, recognised crops.
**Part B** sheet detector: the same steps, then graph recovery.

Models saved earlier are restored right after setup (Colab: from the Google Drive models folder, found by its folder id), so either part can start from saved models (`COMP_EPOCHS = 0`, `SLD_EPOCHS = 0`). Each model is copied back to Drive as soon as it is trained.

Datasets are found by content (files and folders), never by owner or dataset name. Code lives in the repo (`src/Object Detection/nbkit/`); each step below is a short call followed by a check. GPU and Internet on.

## Setup

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["YOLO_VERBOSE"] = "False"      # ultralytics' own output off; the notebook shows one live bar
import fnmatch, hashlib, json, random, shutil, subprocess, sys, zipfile
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import FileLink, display
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
# Unified Kaggle / Colab. Colab: Drive is mounted once, here (HAVE_DRIVE); where drive.mount is not supported, the 3 dataset zips
# are downloaded with PyDrive2 instead. The zips are unpacked under /content/inputs. Kaggle: nothing changes
ON_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
WORK = Path("/content") if ON_COLAB else Path("/kaggle/working")
DRIVE_ROOT, MY_DRIVE = Path("/content/drive"), Path("/content/drive/MyDrive")
DRIVE_ZIPS = {"component-symbols": "component-symbols-v4*.zip",      # Data/releases/2026-10-04
              "component-symbols-test-printed": "component-symbols-test-printed.zip",
              "sld-sheets-v3": "sld-sheets-v3*.zip"}      # also matches "sld-sheets-v3 (2).zip"
HAVE_DRIVE = False
_GDRIVE = None


def pydrive():
    """PyDrive2 client for runtimes without drive.mount; signs in once per session."""
    global _GDRIVE
    if _GDRIVE is None:
        try:
            import pydrive2  # noqa: F401
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyDrive2"], check=True)
        from pydrive2.auth import GoogleAuth
        from pydrive2.drive import GoogleDrive
        from google.colab import auth
        from oauth2client.client import GoogleCredentials
        auth.authenticate_user()
        gauth = GoogleAuth()
        gauth.credentials = GoogleCredentials.get_application_default()
        _GDRIVE = GoogleDrive(gauth)
    return _GDRIVE


def find_drive_zip(pattern):
    """Newest file matching `pattern` in My Drive, at most three folders deep (no walk over all of Drive), never in the trash."""
    hits = [q for pre in ("", "*/", "*/*/", "*/*/*/") for q in MY_DRIVE.glob(pre + pattern)
            if q.is_file() and not any(part.startswith(".Trash") for part in q.parts)]
    return max(hits, key=lambda q: q.stat().st_mtime, default=None)


def pydrive_zip(pattern, out_dir):
    """No-mount fallback: newest Drive file whose title matches `pattern`, downloaded to out_dir."""
    stem = pattern.split("*")[0].removesuffix(".zip")
    items = pydrive().ListFile({"q": f"title contains '{stem}' and trashed=false"}).GetList()
    items = [i for i in items if fnmatch.fnmatch(i["title"], pattern)]
    if not items:
        return None
    item = max(items, key=lambda i: i["modifiedDate"])
    print(f"  Drive file {item['title']}  {int(item.get('fileSize', 0)) / 1e6:.0f} MB  modified {item['modifiedDate'][:16]}")
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / item["title"]
    if not (out.exists() and out.stat().st_size == int(item.get("fileSize", -1))):
        item.GetContentFile(str(out))
    return out


def unpack(zip_path, dest):
    """Unzip into dest.part, then rename: an interrupted unzip never looks like a finished dataset."""
    part = dest.with_name(dest.name + ".part")
    shutil.rmtree(part, ignore_errors=True)
    part.mkdir(parents=True)
    rc = subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", str(part)]).returncode
    if rc not in (0, 1):      # 1 = warnings only (e.g. odd file names); the files are there
        raise RuntimeError(f"unzip {zip_path} failed with exit code {rc}")
    shutil.rmtree(dest, ignore_errors=True)
    part.rename(dest)


if ON_COLAB:
    try:
        from google.colab import drive
        if not MY_DRIVE.is_dir():
            drive.mount(str(DRIVE_ROOT))
    except NotImplementedError:
        print("drive.mount is not supported in this runtime: using PyDrive2 (a sign-in prompt follows)")
    HAVE_DRIVE = MY_DRIVE.is_dir()
    print("Drive mounted at", DRIVE_ROOT if HAVE_DRIVE else "- (not mounted; PyDrive2)")
    COLAB_INPUT = Path("/content/inputs")
    for folder_, pattern_ in DRIVE_ZIPS.items():
        dest_ = COLAB_INPUT / folder_
        if dest_.is_dir() and any(dest_.iterdir()):
            print(f"{folder_}: already unpacked")
            continue
        if HAVE_DRIVE:
            zip_ = find_drive_zip(pattern_)
            if zip_:
                st_ = zip_.stat()
                print(f"{folder_}: {zip_}  {st_.st_size / 1e6:.0f} MB  modified {datetime.fromtimestamp(st_.st_mtime):%Y-%m-%d %H:%M}")
        else:
            print(f"{folder_}:")
            zip_ = pydrive_zip(pattern_, WORK / "drive_zips")
        if zip_ is None:
            raise FileNotFoundError(f"{folder_}: no file matching {pattern_!r} in My Drive "
                                    f"({'searched three folders deep' if HAVE_DRIVE else 'PyDrive2 title search'}). "
                                    "Upload the dataset zip to My Drive and run this cell again.")
        unpack(zip_, dest_)

REPO_URL = "https://github.com/ishuu19/Single-Line-Diagram-and-Energy-Mapping.git"
REPO = WORK / "Single-Line-Diagram-and-Energy-Mapping"
COMP_DIR = REPO / "src" / "Object Detection" / "Component Training"
SLD_DIR = REPO / "src" / "Object Detection" / "SLD Training"
SAVE_DIR = WORK / "sld_pipeline"
INPUT = COLAB_INPUT if ON_COLAB else Path("/kaggle/input")

COMP_EPOCHS = 30     # Part A trains a NEW model ("v3") and never overwrites custom_best; 0 = reuse a saved classifier
REPO_REF = None      # pin the repo to a commit / tag for a reproducible run, e.g. "e572ab1"; None = latest main
SLD_EPOCHS =20      # Part B; 0 = restore saved detector weights (2 = smoke run)
EVAL_N = 150         # test sheets scored at the end; 0 = all
rng = random.Random(7)

In [ ]:
if (REPO / ".git").is_dir():
    if REPO_REF is None:
        subprocess.run(["git", "checkout", "-q", "main"], cwd=REPO)      # an earlier pinned run leaves a detached HEAD that cannot pull
        git_ = subprocess.run(["git", "pull", "-q", "--ff-only"], cwd=REPO, capture_output=True, text=True)
    else:
        git_ = subprocess.run(["git", "fetch", "-q"], cwd=REPO, capture_output=True, text=True)
    print(f"git {'pull' if REPO_REF is None else 'fetch'}: exit {git_.returncode}", (git_.stdout + git_.stderr).strip())
    if git_.returncode:
        print("  WARNING: update failed; running the code already on disk")
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
if REPO_REF:
    subprocess.run(["git", "checkout", "-q", REPO_REF], cwd=REPO, check=True)
print("repo at", subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip())
# modules imported before the pull would keep the old code: drop nbkit and the repo's training modules
STALE_ = {"nbkit", "config", "data", "model", "train", "evaluate", "train_sld", "sld_data", "predict_graph", "reclassify"}
for m_ in [k for k, v in list(sys.modules.items()) if k.split(".")[0] in STALE_ or str(REPO) in str(getattr(v, "__file__", "") or "")]:
    sys.modules.pop(m_, None)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "scikit-image", "networkx"], check=True)
sys.path.insert(0, str(REPO / "src" / "Object Detection"))
import nbkit as nb
nb.ok(COMP_DIR.is_dir() and SLD_DIR.is_dir() and (REPO / "src/Object Detection/nbkit").is_dir(), "repo cloned, nbkit found")

In [ ]:
# Models saved earlier (an earlier session, Part A or B) are restored from the Drive models folder now, before Part A, so
# COMP_EPOCHS = 0 and SLD_EPOCHS = 0 work from the start. The folder is the shared one (by folder id), else My Drive/sld_models.
# Uses the Drive mount from setup (HAVE_DRIVE); without it the folder is read and written with PyDrive2. Kaggle: skipped.
DRIVE_FOLDER_ID = "1F3PIpLIlJIJ0oW191AfEVt6kFlCVuOtW"      # the shared models folder
MODEL_FILES = ("*.keras", "*.names.json", "class_names.json", "sld_*.pt")
DRIVE_MODELS = None


def is_smoke(epochs):
    """1-19 epochs is a smoke run: its models are never copied to Drive (they would replace good ones)."""
    return 0 < epochs < 20


def classifier_files(model_dir, tag):
    """The selected classifier only: <tag>.keras, its <tag>.names.json if present, and class_names.json."""
    return [Path(model_dir) / f"{tag}.keras", Path(model_dir) / f"{tag}.names.json", Path(model_dir) / "class_names.json"]


def drive_download_models(folder_id, dest):
    """No-mount fallback: download the model files under a Drive folder id with PyDrive2."""
    gdrive = pydrive()
    dest.mkdir(parents=True, exist_ok=True)

    def walk(fid):
        for item in gdrive.ListFile({"q": f"'{fid}' in parents and trashed=false", "supportsAllDrives": True,
                                     "includeItemsFromAllDrives": True}).GetList():
            if item["mimeType"] == "application/vnd.google-apps.folder":
                yield from walk(item["id"])
            elif any(fnmatch.fnmatch(item["title"], pat) for pat in MODEL_FILES):
                yield item

    for item in walk(folder_id):
        out = dest / item["title"]
        if not out.exists():
            item.GetContentFile(str(out))
            print("  downloaded", item["title"])
    return dest


def save_to_drive(paths):
    """Copy model files to the Drive models folder (mounted Drive; else uploaded into DRIVE_FOLDER_ID with PyDrive2)."""
    paths = [Path(p) for p in paths if p and Path(p).is_file()]
    if not ON_COLAB:
        print("not on Colab: nothing copied to Drive (use the download link)")
        return []
    if not paths:
        print("no model files to copy to Drive")
        return []
    if HAVE_DRIVE:
        target = DRIVE_MODELS or MY_DRIVE / "sld_models"
        target.mkdir(parents=True, exist_ok=True)
        for p in paths:
            shutil.copy(p, target / p.name)
    else:
        target = f"Drive folder id {DRIVE_FOLDER_ID}"
        gdrive = pydrive()
        for p in paths:
            same = gdrive.ListFile({"q": f"title = '{p.name}' and '{DRIVE_FOLDER_ID}' in parents and trashed=false",
                                    "supportsAllDrives": True, "includeItemsFromAllDrives": True}).GetList()
            if same and same[0].get("md5Checksum") == hashlib.md5(p.read_bytes()).hexdigest():
                continue      # already there, unchanged
            f = same[0] if same else gdrive.CreateFile({"title": p.name, "parents": [{"id": DRIVE_FOLDER_ID}]})
            f.SetContentFile(str(p))
            f.Upload(param={"supportsAllDrives": True})
    print(f"copied to {target}:", sorted(p.name for p in paths))
    return [p.name for p in paths]


if ON_COLAB:
    if HAVE_DRIVE:
        cands_ = [c for c in (DRIVE_ROOT / ".shortcut-targets-by-id" / DRIVE_FOLDER_ID, MY_DRIVE / "sld_models") if c.is_dir()]
        # a shortcut-by-id folder holds the shared folder itself, one level down
        cands_ = [next((d for d in c.iterdir() if d.is_dir()), c) if c.name == DRIVE_FOLDER_ID else c for c in cands_]
        DRIVE_MODELS = next((c for c in cands_ if any(f for pat in MODEL_FILES for f in c.rglob(pat))), None) \
            or (cands_[0] if cands_ else MY_DRIVE / "sld_models")
        DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
    else:
        DRIVE_MODELS = drive_download_models(DRIVE_FOLDER_ID, WORK / "drive_models")
    print("Drive models folder:", DRIVE_MODELS)
    for pat_ in MODEL_FILES:
        for f_ in sorted(DRIVE_MODELS.rglob(pat_)):
            sub_ = "sld" if f_.suffix == ".pt" else "component"
            (SAVE_DIR / "models" / sub_).mkdir(parents=True, exist_ok=True)
            if not (SAVE_DIR / "models" / sub_ / f_.name).exists():
                shutil.copy(f_, SAVE_DIR / "models" / sub_ / f_.name)
                print("  restored", sub_, "/", f_.name)
else:
    print("not on Colab: models come from SAVE_DIR / attached datasets")

## Find the data (by content, no usernames)

In [ ]:
D = nb.io.locate(INPUT)
CROPS, PRINTED, SHEETS = D["crops"], D["printed"], D["sheets"]

In [ ]:
nb.ok(CROPS is not None, f"component crops: {CROPS}")
nb.ok(PRINTED is not None, f"printed test crops: {PRINTED}")
nb.ok(SHEETS is not None, f"full sheets: {SHEETS}")

# Part A - component classifier

## Load the crops

In [ ]:
import tensorflow as tf
for g_ in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g_, True)      # TensorFlow must not take the whole GPU: PyTorch needs it in Part B
for m_ in ("config", "data", "model", "train", "evaluate"):
    sys.modules.pop(m_, None)
sys.path.insert(0, str(COMP_DIR))
import config as ccfg, data as cdata, model as cmodel, train as ctrain

nb.io.link(CROPS, REPO / "Data" / "component-symbols")
nb.io.link(PRINTED, REPO / "Data" / "component-symbols-test-printed")
man, CLASS_NAMES = nb.classifier.load_manifest(CROPS)

In [ ]:
nb.ok(len(man) > 0 and set(man.split) >= {"train", "test"}, f"{len(man):,} crops: train {int((man.split == 'train').sum()):,}, test {int((man.split == 'test').sum()):,} (test is never used for early stopping)")
nb.ok(man.file.is_unique, "no duplicate manifest rows (duplicates would double-count crops in training)")
nb.ok(len(CLASS_NAMES) > 0 and man.type.nunique() >= len(CLASS_NAMES), f"{man.type.nunique()} symbol types with crops, "
      f"{len(CLASS_NAMES)} in classes.txt" + (f"; no crops for {sorted(set(CLASS_NAMES) - set(man.type))}" if set(CLASS_NAMES) - set(man.type) else ""))
print(f"v3 crops: {int(man.v3.sum()):,}" + ("" if man.v3.any() else "  (none: re-upload the crop dataset with the _v3_ files)"))
SHEETS_HAVE_V3 = nb.io._sheets_info(SHEETS)[0]
print(f"v3 sheets: {'yes' if SHEETS_HAVE_V3 else 'no'}" + ("" if man.v3.any() == SHEETS_HAVE_V3 else "   <- crops and sheets are not at the same version"))

## Look at the dataset

In [ ]:
nb.classifier.plot_overview(man)

In [ ]:
nb.classifier.plot_same_type(man, CROPS, CLASS_NAMES, rng)

## Augmentation, one effect at a time

In [ ]:
n_steps = nb.classifier.plot_aug_steps(CROPS, man, rng, ccfg, cdata, cmodel)

In [ ]:
nb.ok(n_steps == 10, "augmentation steps drawn: rotate, zoom, shift, contrast, thicker lines, blur, noise, blank patch, whole pipeline")

## Build data and model

In [ ]:
ctrain.setup()
hp = ctrain.load_hp(None)
train_ds, val_ds, class_names, class_weights = cdata.load_datasets(mixup_alpha=hp["mixup_alpha"])

In [ ]:
nb.ok(int(train_ds.cardinality()) > 0 and int(val_ds.cardinality()) > 0, f"batches: train {int(train_ds.cardinality())}, validation {int(val_ds.cardinality())} (carved from train; the test split stays untouched)")
nb.ok(len(class_names) == len(set(man.type)), f"{len(class_names)} classifier classes match the manifest")

In [ ]:
if COMP_EPOCHS:
    net = cmodel.build_model(len(class_names), hp)
    steps = int(train_ds.cardinality())
    cmodel.compile_model(net, ctrain.lr_schedule(steps, hp["learning_rate"], max(COMP_EPOCHS, ccfg.WARMUP_EPOCHS + 1)), hp)
    nb.ok(net.count_params() > 0, f"model built, {net.count_params():,} parameters")
else:
    print("COMP_EPOCHS = 0: nothing to build")

## Train (live progress bars)

In [ ]:
HIST = SAVE_DIR / "outputs" / "component" / "history.json"
if COMP_EPOCHS:
    hist = net.fit(train_ds, validation_data=val_ds, epochs=COMP_EPOCHS, class_weight=class_weights, verbose=0,
                   callbacks=ctrain.callbacks("v3") + [nb.classifier.live_bar()]).history
    net.save(str(ccfg.MODEL_DIR / "v3.keras"))
    (ccfg.MODEL_DIR / "class_names.json").write_text(json.dumps(class_names))
    HIST.parent.mkdir(parents=True, exist_ok=True)
    HIST.write_text(json.dumps(hist))
else:
    print("COMP_EPOCHS = 0: training skipped")

In [ ]:
if COMP_EPOCHS and is_smoke(COMP_EPOCHS):
    print(f"smoke run: best val accuracy {max(hist['val_accuracy']):.3f} after {len(hist['loss'])} epochs (not checked)")
elif COMP_EPOCHS:      # a full run must reach a usable classifier
    nb.ok(max(hist["val_accuracy"]) >= 0.8, f"best val accuracy {max(hist['val_accuracy']):.3f} after {len(hist['loss'])} epochs (>= 0.8)")

## Restore the best classifier

In [ ]:
saved = nb.io.restore(["v3_best.keras", "custom_best.keras", "class_names.json"], ccfg.MODEL_DIR,
                      [SAVE_DIR / "models" / "component", *{p.parent for p in D["found"]["keras"]}])
COMP_TAG = next((t for t in ("v3_best", "custom_best", "custom", "notebook_best") if t in {p.stem for p in ccfg.MODEL_DIR.glob("*.keras")}), None)
nb.ok(COMP_TAG is not None, f"classifier file found: {COMP_TAG}  (models folder: {saved})")
NAMES = json.loads((ccfg.MODEL_DIR / "class_names.json").read_text()) if (ccfg.MODEL_DIR / "class_names.json").exists() else class_names

In [ ]:
nb.ok(len(NAMES) == len(class_names), f"{len(NAMES)} class names match the dataset; candidate checkpoint: {COMP_TAG}")

## Learning curves: overfitting or underfitting?

In [ ]:
if HIST.exists():
    verdict = nb.classifier.plot_curves(pd.DataFrame(json.loads(HIST.read_text())))
else:
    print("no training history in this session (the classifier was restored)")

## Test on held-out crops

In [ ]:
sets = {
    "train sample": [CROPS / man.file[i] for i in nb.classifier.draw(man[man.split == "train"].index, 800, rng)],
    "test split (same styles)": [CROPS / f for f in man[man.split == "test"].file],
    "printed test (unseen styles)": sorted((PRINTED / "images" / "test").glob("*.png")),
}
labels = {str(CROPS / f): t for f, t in zip(man.file, man.type)}
labels.update({str(PRINTED / r["file"]): r["type"] for r in map(json.loads, (PRINTED / "manifest.jsonl").open()) if r.get("type")})
sets = {k: [str(p) for p in v] for k, v in sets.items()}
# score every classifier we have; the better one on the held-out sets is used from here on (a new run cannot replace a better old one)
scored = {}
for tag in [t for t in ("v3_best", "custom_best") if (ccfg.MODEL_DIR / f"{t}.keras").exists()] or [COMP_TAG]:
    m_ = tf.keras.models.load_model(ccfg.MODEL_DIR / f"{tag}.keras", compile=False)
    if m_.output_shape[-1] == len(NAMES):
        scored[tag] = (m_, nb.classifier.score_sets(m_, NAMES, sets, labels=labels))
held_out = lambda res: sum(float((r["pred"] == r["y"]).mean()) for n, r in res.items() if n != "train sample") / max(1, len(res) - 1)
COMP_TAG = max(scored, key=lambda t: held_out(scored[t][1]))
clf, R = scored[COMP_TAG]
display(pd.DataFrame({t: {n: float((r["pred"] == r["y"]).mean()) for n, r in res.items()} for t, (_, res) in scored.items()}).round(3))
print("using:", COMP_TAG)

In [ ]:
nb.ok(len(R) == 3 and all(len(r["y"]) > 0 for r in R.values()), "scored: " + ", ".join(f"{k} ({len(r['y'])})" for k, r in R.items()))
nb.ok(clf.output_shape[-1] == len(NAMES), f"selected {COMP_TAG}: {clf.output_shape[-1]} outputs = {len(NAMES)} class names")

In [ ]:
acc = nb.classifier.plot_accuracy(R)
full = COMP_EPOCHS == 0 or COMP_EPOCHS >= 20          # thresholds only mean something for a real run, not a 2-epoch smoke test
nb.ok(not full or acc["test split (same styles)"] >= 0.90, f"same-style test accuracy {acc['test split (same styles)']:.3f} (needs >= 0.90)")
nb.ok(not full or acc["printed test (unseen styles)"] >= 0.50, f"unseen-style accuracy {acc['printed test (unseen styles)']:.3f} (needs >= 0.50)")

### ROC curves and residuals

In [ ]:
nb.classifier.plot_roc_residuals(R, NAMES)

### How many does it recognise? The images

In [ ]:
nb.classifier.show_recognised(R, NAMES, rng)

## Save Part A

In [ ]:
(SAVE_DIR / "models").mkdir(parents=True, exist_ok=True)
shutil.copytree(ccfg.MODEL_DIR, SAVE_DIR / "models" / "component", dirs_exist_ok=True)
out_a = SAVE_DIR / "outputs" / "component"
out_a.mkdir(parents=True, exist_ok=True)
(out_a / "test_accuracy.json").write_text(json.dumps(acc.to_dict(), indent=1))
nb.ok(any((SAVE_DIR / "models" / "component").glob("*.keras")), "classifier saved under sld_pipeline/models/component")
if COMP_EPOCHS and not is_smoke(COMP_EPOCHS):      # to Drive now, so a later crash or disconnect does not lose it
    save_to_drive(classifier_files(ccfg.MODEL_DIR, COMP_TAG))
else:
    print("classifier not copied to Drive:", "smoke run" if COMP_EPOCHS else "restored, not trained")

# Part B - full sheets

## Import Part B (the saved models were restored from Drive after setup)

In [ ]:
# After this cell Part A cannot be re-run (its modules, models and working folder are replaced); restart the runtime for that.
for name_ in ("clf", "scored", "m_", "net", "train_ds", "val_ds", "hist", "sets", "R", "labels"):     # Part A's models out of memory before the detector needs the GPU
    globals().pop(name_, None)
plt.close("all")
nb.io.free_gpu()
for m_ in ("config", "data", "model", "train", "evaluate"):
    sys.modules.pop(m_, None)
sys.path = [p for p in sys.path if p != str(COMP_DIR)]
os.chdir(SLD_DIR)
sys.path.insert(0, str(SLD_DIR))
import config
from sld_data import sheet_ids
from predict_graph import load_detector
from reclassify import load_classifier
from train_sld import resolve_data_yaml

comp_models = COMP_DIR / "models"
nb.io.restore(["v3_best.keras", "custom_best.keras", "class_names.json"], comp_models, [SAVE_DIR / "models" / "component", *{p.parent for p in D["found"]["keras"]}])
nb.io.restore(["sld_*.pt"], config.MODEL_DIR, [SAVE_DIR / "models" / "sld", *{p.parent for p in D["found"]["pt"]}])
COMP_TAG = globals().get("COMP_TAG") or next(t for t in ("v3_best", "custom_best", "custom", "notebook_best") if (comp_models / f"{t}.keras").exists())
saved_pt = sorted(config.MODEL_DIR.glob("sld_*.pt"))

In [ ]:
nb.ok((comp_models / f"{COMP_TAG}.keras").exists(), f"classifier restored: {COMP_TAG}")
print("detector weights on disk:", [p.name for p in saved_pt] or "none yet (will be trained)")

## Load the sheets

In [ ]:
nb.io.link(SHEETS, config.SHEETS_DIR)
ctx = nb.sheets.make_ctx()

In [ ]:
nb.ok(all(len(v) > 0 for v in ctx.ids.values()), "sheets: " + ", ".join(f"{k} {len(v):,}" for k, v in ctx.ids.items()))
# data.yaml lists only the symbol types present in the corpus (SYMBOL_ORDER order) plus junction and text
nb.ok(len(ctx.classes) >= 20 and ctx.classes[-2:] == [config.JUNCTION_CLASS, config.TEXT_CLASS]
      and set(ctx.classes) <= set(config.SYMBOL_ORDER) | {config.JUNCTION_CLASS, config.TEXT_CLASS},
      f"{len(ctx.classes)} detector classes ({len(ctx.classes) - 2} of {len(config.SYMBOL_ORDER)} symbol types, plus junction and text)")

## Look at the dataset

In [ ]:
n_sheets = nb.sheets.plot_sheet_counts(ctx)

In [ ]:
n_shown = nb.sheets.show_ground_truth(ctx, rng)
nb.ok(n_shown >= 1, f"{n_shown} ground-truth sheets drawn (TB / LR / RL)")

## Augmentation, one effect at a time

In [ ]:
n_panels = nb.sheets.show_augmentation(ctx, rng)
nb.ok(n_panels >= 1, f"{n_panels} augmentation panels drawn on two sheets (the effects training applies at random)")

## Train the detector (live progress bars)

In [ ]:
RUN = SAVE_DIR / "runs" / "sld"
if SLD_EPOCHS:
    # a run folder from an earlier training is deleted first: train_sld reuses runs/sld (exist_ok), ultralytics appends to an old
    # results.csv, and a crashed run would leave the old weights/best.pt looking like this run's
    if RUN.exists():
        shutil.rmtree(RUN)
        print("earlier run folder removed:", RUN)
    WEIGHTS = nb.sheets.train_detector(SLD_EPOCHS, SAVE_DIR / "runs")       # train_sld.train: one code path with the command line
else:
    WEIGHTS = saved_pt[-1] if saved_pt else None

In [ ]:
nb.ok(WEIGHTS is not None and Path(WEIGHTS).exists(), f"detector weights: {WEIGHTS}")

In [ ]:
# Detector to Drive now (Colab), so a later crash or disconnect does not lose it; smoke runs and restored weights are not copied
if SLD_EPOCHS and not is_smoke(SLD_EPOCHS):
    save_to_drive([WEIGHTS])
else:
    print("detector not copied to Drive:", "smoke run" if SLD_EPOCHS else "restored, not trained")

## Learning curves, real augmented batches, PR / F1 curves, confusion matrix

In [ ]:
verdict_b = nb.sheets.plot_detector_curves(RUN)

## Test split

In [ ]:
map50, map5095 = nb.sheets.test_map(WEIGHTS, resolve_data_yaml(config.DATA_YAML, SAVE_DIR / "data.yaml"), config.IMG_SIZE)
need = 0.5 if SLD_EPOCHS == 0 or SLD_EPOCHS >= 20 else 0.0       # a 2-epoch smoke run cannot reach a real threshold
nb.ok(map50 >= need, f"test mAP50 {map50:.3f} (needs >= {need}), mAP50-95 {map5095:.3f}")

### Detector confidence threshold

In [ ]:
detector = load_detector(WEIGHTS)
conf_table, CONF = nb.sheets.conf_sweep(ctx, detector, rng)
print("best F1 at confidence", CONF)

In [ ]:
classifier = load_classifier(comp_models / f"{COMP_TAG}.keras")
recover = nb.sheets.Recoverer(ctx, detector, classifier, conf=CONF)
sample = nb.sheets.balanced_sample(ctx, rng, EVAL_N)
nb.ok(len(sample) > 0, f"{len(sample)} test sheets to score at confidence {CONF}; detector and classifier loaded")

### How many symbols does it find? The images

In [ ]:
rows, found = nb.sheets.evaluate_sheets(ctx, recover, sample)
examples = nb.sheets.pick_examples(rows)

In [ ]:
tot_found, tot_gt = nb.sheets.plot_found(rows, found)
nb.ok(tot_gt > 0 and len(examples) >= 1, f"{len(rows)} sheets scored; typical example per direction: {examples}")

In [ ]:
nb.sheets.show_found_sheets(ctx, recover, examples)

## Where does the error come from?
Three stages on the same sheets: the tracer alone (true boxes), the detector, then the Part A classifier. The detector stages are scored **by location**, so a finer classifier label is not counted as a missed node; `node_type_accuracy` is how often the matched box also has the right type.

In [ ]:
stages = nb.sheets.stage_table(rows)
display(stages)
nb.ok(len(stages) in (12, 15), f"stage table: {len(stages) // 3} groups (all, TB, LR, RL, plus other when PLANT sheets are scored) x 3 stages")

In [ ]:
nb.sheets.plot_scores(rows)

In [ ]:
display(nb.sheets.typed_vs_loc(rows))        # the same sheets under the strict type-gated rule vs location-only

## Recovered graphs and the edge audit

In [ ]:
nb.sheets.show_recovered(ctx, recover, examples)

In [ ]:
nb.sheets.show_edge_audit(ctx, recover, examples)     # green right, red wrong type, orange dashed missed, purple extra

In [ ]:
nb.sheets.show_guess_vs_true(ctx, recover, examples)

In [ ]:
nb.sheets.show_worst(ctx, rows, recover)

## Save and download

In [ ]:
config.OUTPUT_DIR.mkdir(exist_ok=True)
(config.OUTPUT_DIR / "graph_pred_test.json").write_text(json.dumps({
    "confidence": CONF, "classifier": COMP_TAG, "map50": map50, "map50_95": map5095,
    "stages": {f"{d} | {st}": row.to_dict() for (d, st), row in stages.iterrows()},
    "symbols_found": {"found": tot_found, "ground_truth": tot_gt},
    "sheets": {r["sheet"]: r["clf_loc"] for r in rows}}, indent=1))
for src, sub in ((config.MODEL_DIR, "models/sld"), (config.OUTPUT_DIR, "outputs/sld")):
    shutil.copytree(src, SAVE_DIR / sub, dirs_exist_ok=True)

In [ ]:
# Copy the selected classifier and the detector to the Drive models folder (Colab) before the download; smoke runs are skipped
files_ = []
if globals().get("COMP_TAG") and not is_smoke(COMP_EPOCHS):
    files_ += classifier_files(COMP_DIR / "models", COMP_TAG)
if globals().get("WEIGHTS") and not is_smoke(SLD_EPOCHS):
    files_.append(Path(WEIGHTS))
save_to_drive(files_)

In [ ]:
zip_path = nb.io.bundle(SAVE_DIR, {"runs/sld": RUN}, [SLD_DIR / "notebooks" / "kaggle-sld-pipeline-v3.ipynb"],
                        str(WORK / "sld_pipeline_artifacts"), str(WORK / "sld_pipeline_artifacts.zip"))
with zipfile.ZipFile(zip_path) as zf_:
    models_in_zip_ = [n for n in zf_.namelist() if n.endswith((".pt", ".keras"))]
nb.ok(len(models_in_zip_) > 0, f"bundle written: {Path(zip_path).stat().st_size / 1e6:.0f} MB, {len(models_in_zip_)} model files "
      "(models, outputs and runs/sld; no datasets)")
if ON_COLAB:
    from google.colab import files
    files.download(str(zip_path))
else:
    display(FileLink(str(zip_path), result_html_suffix=" - download zip"))